# WO-08 — WO-07 decisions, six new units, and the coastline
Cells 3 to 5 give the cutoff evidence for S1.3 (distance limit 25, 50, 100 km) and S1.4 (share 5, 10, 20%). Wording is `profile_templates.md`; rules are in `scripts/edop/whg/` (`wo08_units.py` for the new units). Results: `docs/design/dashboard/wo08_results.md`. Read-only except `wo08_filled_profiles.md` and the CSVs under `output/edop/whg/`.

In [ ]:
%matplotlib inline
# Cell 1 — Setup: wording file, server check, saved ring, flow and river tables
import sys, time, warnings
warnings.filterwarnings("ignore", message="pandas only supports SQLAlchemy")
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'scripts' / 'edop').exists())
sys.path.insert(0, str(ROOT))
from scripts.shared.db_utils import db_connect
from scripts.edop.whg import wo03_common as C
from scripts.edop.whg import wo04_fill as W
from scripts.edop.whg import wo06_rings as R
from scripts.edop.whg import wo07_rivers as RV
from scripts.edop.whg import wo08_units as U8

conn = db_connect()
PAIRS = pd.read_csv(C.OUT / 'q1_pairs.csv', dtype={'tag': str})
FLOW = pd.read_csv(C.OUT / 'q2_flow_pairs.csv', dtype={'tag': str})
SAMPLE = C.population_points()
RING = R.load_table()
RIV = RV.load_table() if RV.RIVERS_CSV.exists() else None
print('commit', C.commit(), '| server:', W.get_json('/api/health'), '| wording:', W.templates_stamp(), '| ring table:', len(RING), 'basins | river table:', len(RIV) if RIV else 'not built yet', '| test places:', len(C.PLACES), '| sample points:', len(SAMPLE))

In [ ]:
# Cell 2 — The ten test places: the filled_profiles (written to docs/design/dashboard/wo08_filled_profiles.md) and the sentences of the new units
OUTMD = ROOT / 'docs' / 'design' / 'dashboard' / 'wo08_filled_profiles.md'
t0 = time.perf_counter()
TEST = {}
parts = ['# WO-08: the dummy filled from real data, ten test places (filled_profiles)', '',
         'Generated by `notebooks/edop/whg/wo08_new_units.ipynb` (commit ' + C.commit() + '). Every `yes` unit of `modal_dummy_annotated.md` is filled from data; each `maybe` or `no` unit is a placeholder in the form `⟦id maybe: needs ...⟧`. Units with nothing true to say are silent. Wording is the template file\'s, not polished.', '',
         'Wording from `' + W.templates_stamp() + '`.', '']
L = []
for name, (lat, lon) in C.PLACES.items():
    F = W.build_facts(lat, lon, conn, PAIRS, FLOW, ring=RING, rivers=RIV)
    if F['pub'] is None:
        L.append(f'{name}: no basin'); continue
    Un = W.fill_units(F)
    TEST[name] = (F, Un)
    parts.append(W.render_filled_profile(name, F, Un)); parts.append('---'); parts.append('')
    L.append(name)
    for u in ('S1.3', 'S1.4', 'S2.1b', 'S2.3', 'S3.4', 'S5.2'):
        if u in Un:
            L.append(f'   {u}: ' + (Un[u]['text'] if Un[u]['ok'] else f'[silent: {Un[u]["why"]}]'))
OUTMD.write_text('\n'.join(parts))
print(f'wrote {OUTMD.name} | {W.templates_stamp()} | {time.perf_counter() - t0:.0f}s\n\n' + '\n'.join(L))

In [ ]:
# Cell 3 — B3: distance to the next ecoregion for the ten places and the 306 sample points; saved to a CSV (resumable; about 1 to 2 minutes)
OUT3 = C.OUT / 'wo08_next_eco.csv'
done = pd.read_csv(OUT3, dtype={'key': str}) if OUT3.exists() else pd.DataFrame(columns=['key'])
have = set(done['key'])
pts = ([(n, 'test', la, lo) for n, (la, lo) in C.PLACES.items()]
       + [(str(int(s['hybas_id_l6'])), 'sample', float(s['lat']), float(s['lon'])) for s in SAMPLE])
inv = U8.invalid_ids(conn)
rows, t0 = [], time.perf_counter()
for k, (key, grp, la, lo) in enumerate(pts, 1):
    if key in have:
        continue
    pe = conn.execute('SELECT e.eco_id::bigint FROM gaz."Ecoregions2017" e WHERE e.geom && ST_SetSRID(ST_MakePoint(%s, %s), 4326) AND ST_Intersects(CASE WHEN e.eco_id = ANY(%s) THEN ST_MakeValid(e.geom) ELSE e.geom END, ST_SetSRID(ST_MakePoint(%s, %s), 4326)) LIMIT 1', (lo, la, inv, lo, la)).fetchone()
    b = conn.execute('SELECT hybas_id::bigint FROM public.basin06 WHERE ST_Intersects(geom, ST_SetSRID(ST_MakePoint(%s, %s), 4326)) LIMIT 1', (lo, la)).fetchone()
    top = U8.eco_shares(conn, b[0])[:1] if b else []
    own = int(pe[0]) if pe else (top[0][0] if top else None)
    nx = U8.next_ecoregion(conn, la, lo, own) if own is not None else None
    rows.append(dict(key=key, group=grp, lat=la, lon=lo, hid=b[0] if b else None, own=own, own_from='point' if pe else ('basin' if top else ''),
                     basin_top=top[0][0] if top else None, next_id=nx[0] if nx else None, next_name=nx[1] if nx else None, km=nx[2] if nx else None))
    if len(rows) % 50 == 0:
        print(f'  {len(rows)} done in {time.perf_counter() - t0:.0f}s')
if rows:
    done = pd.concat([done, pd.DataFrame(rows)], ignore_index=True)
    done.to_csv(OUT3, index=False)
print(f'rows saved: {len(done)} | this run {len(rows)} in {time.perf_counter() - t0:.0f}s')

In [ ]:
# Cell 4 — B3 report: how often a next ecoregion lies within D km, and how often it is the ecoregion S1.2c already names
N3 = pd.read_csv(C.OUT / 'wo08_next_eco.csv', dtype={'key': str})
S3 = N3[(N3['group'] == 'sample') & N3['hid'].notna()]
n = len(S3)
L = [f'sample points with a basin: {n} | with an ecoregion to measure from: {int(S3["own"].notna().sum())} | point in no ecoregion polygon (basin majority used): {int((S3["own_from"] == "basin").sum())}', '']
L.append('distance from the point to the nearest other ecoregion, km: ' + str(S3['km'].describe(percentiles=[.1, .25, .5, .75, .9]).round(1).to_dict()))
L.append('')
for D in (25, 50, 100):
    k = int((S3['km'] <= D).sum())
    L.append(f'within {D:>3} km: the sentence would speak at {k} of {n} ({100 * k / n:.0f}%); silent at {n - k}')
L.append(f'within 1 km ("less than a kilometre"): {int((S3["km"] < 1).sum())}')
same = S3[S3['next_id'] == S3['basin_top']]
L.append('')
L.append(f'the nearest other ecoregion is also the basin\'s largest-share ecoregion (so S1.2c, where it speaks, names it too): {len(same)} of {n}')
L.append('')
L.append('the ten places:')
T3 = N3[N3['group'] == 'test'][['key', 'own', 'own_from', 'next_name', 'km']].copy()
T3['km'] = T3['km'].round(1)
L.append(T3.to_string(index=False))
print('\n'.join(L))

In [ ]:
# Cell 5 — B4: the basin's other ecoregions at 5, 10 and 20% (shares from WO-03's q3_basin_eco.csv; the point's ecoregion from the table of Cell 3)
EB = pd.read_csv(C.OUT / 'q3_basin_eco.csv')
EB = EB[EB['eco_id'] >= 0]
NAME = dict(conn.execute('SELECT eco_id::bigint, eco_name FROM gaz."Ecoregions2017"').fetchall())
EB['share'] = EB['km2'] / EB['basin_km2']
N3 = pd.read_csv(C.OUT / 'wo08_next_eco.csv', dtype={'key': str})
rows = []
for r in N3[N3['hid'].notna()].itertuples():
    sub = EB[(EB['basin_id'] == int(r.hid)) & (EB['share'] > 0)].sort_values('share', ascending=False)
    if sub.empty:
        rows.append(dict(key=r.key, group=r.group, have_shares=False)); continue
    top = int(sub['eco_id'].iloc[0])
    skip = {top, int(r.own) if pd.notna(r.own) else None}
    oth = sub[~sub['eco_id'].isin(skip)]
    d = dict(key=r.key, group=r.group, have_shares=True)
    for t in (5, 10, 20):
        d[f'n{t}'] = int((oth['share'] >= t / 100).sum())
    d['names10'] = ', '.join(f"{NAME[int(e)]} {s:.0%}" for e, s in zip(oth['eco_id'], oth['share']) if s >= .10)
    d['names5'] = ', '.join(f"{NAME[int(e)]} {s:.0%}" for e, s in zip(oth['eco_id'], oth['share']) if s >= .05)
    rows.append(d)
B4 = pd.DataFrame(rows)
B4.to_csv(C.OUT / 'wo08_other_eco.csv', index=False)
S4 = B4[(B4['group'] == 'sample') & B4['have_shares']]
n = len(S4)
L = [f'sample basins with ecoregion shares: {n} of {int((B4["group"] == "sample").sum())}', '']
for t in (5, 10, 20):
    vc = S4[f'n{t}'].clip(upper=3).value_counts().sort_index()
    L.append(f'at {t:>2}%: other ecoregions per basin (0, 1, 2, 3 or more): ' + str({int(k): int(v) for k, v in vc.items()}) + f' | the sentence would speak at {int((S4[f"n{t}"] > 0).sum())} of {n} ({100 * (S4[f"n{t}"] > 0).sum() / n:.0f}%)')
L.append('')
L.append('the ten places (count at 5% / 10% / 20%, then names at 10%):')
for r in B4[B4['group'] == 'test'].itertuples():
    L.append(f"   {r.key}: {getattr(r, 'n5', '-')} / {getattr(r, 'n10', '-')} / {getattr(r, 'n20', '-')} | {getattr(r, 'names10', '')}")
print('\n'.join(L))